In [ ]:
!pip install --upgrade torchvision

In [ ]:
import torchvision
print(torchvision.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q '/content/drive/MyDrive/VisDrone2019-DET-train.zip' -d /content/visdrone

In [ ]:
!unzip -q '/content/drive/MyDrive/VisDrone2019-DET-val.zip' -d /content/visdroneval

In [ ]:
!unzip -q '/content/drive/MyDrive/VisDrone2019-DET-test-dev.zip' -d /content/visdrontest

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class VisDroneDataset(Dataset):
    def __init__(self, images_path, annotations_path, transforms=None):
        self.images_path = images_path
        self.annotations_path = annotations_path
        self.transforms = transforms
        self.imgs = list(sorted(os.listdir(images_path)))
        self.labels = list(sorted(os.listdir(annotations_path)))

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_path, self.imgs[idx])
        label_path = os.path.join(self.annotations_path, self.labels[idx])

        img = Image.open(img_path).convert("RGB")
        original_size = img.size
        img = img.resize((400, 400))  # Resize the image to 400x400

        labels = self.parse_annotations(label_path, original_size, (400, 400))

        if self.transforms:
            img = self.transforms(img)

        # Prepare targets as a dictionary
        boxes = torch.tensor(labels[:, :4], dtype=torch.float32)
        labels = torch.tensor(labels[:, 4], dtype=torch.int64)
        target = {'boxes': boxes, 'labels': labels}

        return img, target

    def __len__(self):
        return len(self.imgs)

    def parse_annotations(self, annotation_path, original_size, new_size):
        original_width, original_height = original_size
        new_width, new_height = new_size
        labels = []

        with open(annotation_path, 'r') as file:
            for line in file.readlines():
                bbox_left, bbox_top, bbox_width, bbox_height, score, object_category = map(int, line.strip().split(',')[:6])
                if object_category == 0 or bbox_width <= 0 or bbox_height <= 0 or score == 0 or object_category > 10:
                    continue  # Ignore if the category is ignored regions or bbox dimensions are non-positive

                # Scale bounding boxes to the new image size
                new_bbox_left = bbox_left * new_width / original_width
                new_bbox_top = bbox_top * new_height / original_height
                new_bbox_right = (bbox_left + bbox_width) * new_width / original_width
                new_bbox_bottom = (bbox_top + bbox_height) * new_height / original_height

                scaled_bbox = [
                    new_bbox_left,
                    new_bbox_top,
                    new_bbox_right,
                    new_bbox_bottom,
                    object_category
                ]
                labels.append(scaled_bbox)
        return np.array(labels)

# ImageNet mean and std for normalization
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Define the full paths for images and annotations
images_path = '/content/visdrone/VisDrone2019-DET-train/images'
annotations_path = '/content/visdrone/VisDrone2019-DET-train/annotations'

# Create the dataset
train_dataset = VisDroneDataset(images_path, annotations_path, transforms=transform)

# Create data loader with reduced batch size
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))


In [ ]:
val_images_path = '/content/visdroneval/VisDrone2019-DET-val/images'
val_annotations_path = '/content/visdroneval/VisDrone2019-DET-val/annotations'

val_dataset = VisDroneDataset(val_images_path, val_annotations_path, transforms=transform)

val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))

In [ ]:
test_images_path = '/content/visdrontest/images'
test_annotations_path = '/content/visdrontest/annotations'

test_dataset = VisDroneDataset(test_images_path, test_annotations_path, transforms=transform)

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))

In [ ]:
import torch
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.mobilenetv3 import mobilenet_v3_large
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import os

# Ensure the correct version of torchvision is installed
#!pip install torchvision --upgrade

# Load a pre-trained MobileNetV3 model for feature extraction
backbone = mobilenet_v3_large(weights='IMAGENET1K_V2').features

# Modify the backbone to match the expected input channels for the FasterRCNN model
backbone.out_channels = 960

# Check the number of feature maps produced by the backbone
dummy_input = torch.randn(1, 3, 224, 224)
feature_maps = backbone(dummy_input)
num_feature_maps = len(feature_maps)

print(f"Number of feature maps: {num_feature_maps}")

# Define anchor sizes and aspect ratios for the anchor generator based on the number of feature maps
anchor_sizes = tuple((32 * 2 ** i,) for i in range(num_feature_maps))
aspect_ratios = ((0.5, 1.0, 2.0),) * num_feature_maps

# Create the anchor generator
anchor_generator = torchvision.models.detection.rpn.AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

# Create the FasterRCNN model using the MobileNetV3 backbone and anchor generator
model = FasterRCNN(backbone, num_classes=11, rpn_anchor_generator=anchor_generator)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Initialize GradScaler for mixed precision training
scaler = torch.cuda.amp.GradScaler()

# Define the path to save the model weights
save_path = '/content/drive/MyDrive'

# Create the save directory if it does not exist
if not os.path.exists(save_path):
    os.makedirs(save_path)



In [ ]:
# Training and validation loop
num_epochs = 3
accumulation_steps = 4

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    epoch_loss = 0
    tqdm_loader = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for i, (images, targets) in enumerate(tqdm_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.amp.autocast(device_type='cuda'):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses = losses / accumulation_steps  # Scale the loss

        scaler.scale(losses).backward()

        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += losses.item()
        average_loss = epoch_loss / (i + 1)
        tqdm_loader.set_postfix(loss=average_loss)  # Update the progress bar with the current loss

    lr_scheduler.step()

    # Save the model weights
    model_save_path = os.path.join(save_path, f'model_epoch_{epoch+1+5}.pth')
    torch.save(model.state_dict(), model_save_path)
    print(f'Model weights saved to {model_save_path}')


In [ ]:
import matplotlib.pyplot as plt
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms.functional import to_pil_image
import torch

model = model.to(device)
model.eval()

# Load your trained model weights if needed
model.load_state_dict(torch.load('/content/drive/MyDrive/model_epoch_8.pth'))

# Mean and std for normalization (should match those used in training)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Function to denormalize the image
def denormalize_image(tensor, mean, std):
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(tensor, 0, 1)

# Define a helper function to visualize the predictions
def visualize_predictions(image, boxes, labels, scores, threshold=0.5):
    # Denormalize the image
    image = denormalize_image(image.clone(), mean, std)

    # Convert the tensor image to a PIL image for visualization
    image_pil = to_pil_image(image)
    image_width, image_height = image_pil.size

    plt.figure(figsize=(10, 10))
    plt.imshow(image_pil)

    ax = plt.gca()
    for box, label, score in zip(boxes, labels, scores):
        if score >= threshold:
            # Convert normalized coordinates to absolute pixel values
            xmin, ymin, xmax, ymax = box

            ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                       fill=False, color='red', linewidth=2))
            ax.text(xmin, ymin, f'{label}: {score:.2f}', color='white',
                    fontsize=12, bbox=dict(facecolor='red', alpha=0.5))

    plt.axis('off')
    plt.show()

# Iterate over the dataset
for images, targets in val_loader:
    images = [image.to(device) for image in images]  # Move images to the device

    # Make predictions
    with torch.no_grad():
        predictions = model(images)

    # Visualize each image and its predictions
    for i in range(min(2, len(images))):  # Ensure not to exceed available images
        image = images[i].cpu()  # Move the image back to the CPU for visualization
        prediction = predictions[i]

        boxes = prediction['boxes'].cpu().numpy()
        labels = prediction['labels'].cpu().numpy()
        scores = prediction['scores'].cpu().numpy()

        # Visualize predictions
        visualize_predictions(image, boxes, labels, scores, threshold=0.5)

    # Stop after one batch for demonstration purposes
    break


In [ ]:
import torch
import torchvision

# Ensure your validation data loader (val_loader) returns both images and targets
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass without targets to get predictions
        outputs = model(images)

        # Collect predictions and targets
        for i, output in enumerate(outputs):
            # Move predictions to CPU and store in the all_predictions list
            pred_boxes = output['boxes'].cpu()
            pred_scores = output['scores'].cpu()
            pred_labels = output['labels'].cpu()

            all_predictions.extend([{
                'boxes': box,
                'scores': score,
                'labels': label
            } for box, score, label in zip(pred_boxes, pred_scores, pred_labels)])

            # Move targets to CPU and store in the all_targets list
            target_boxes = targets[i]['boxes'].cpu()
            target_labels = targets[i]['labels'].cpu()

            all_targets.extend([{
                'boxes': box,
                'labels': label
            } for box, label in zip(target_boxes, target_labels)])

# Now `all_predictions` and `all_targets` contain the predictions and ground truth for the entire validation set.


In [ ]:
import torch

def apply_nms(predictions, iou_threshold=0.5):
    nms_predictions = []

    for pred in predictions:
        # Convert boxes to tensor
        boxes = torch.stack([p['boxes'] for p in pred])
        scores = torch.stack([p['scores'] for p in pred])
        labels = torch.stack([p['labels'] for p in pred])

        # Perform NMS
        keep = torch.ops.torchvision.nms(boxes, scores, iou_threshold)

        # Collect NMS-filtered predictions
        nms_predictions.append([{
            'boxes': boxes[i],
            'scores': scores[i],
            'labels': labels[i]
        } for i in keep])

    return nms_predictions

# Apply NMS on predictions
nms_predictions = apply_nms(all_predictions)


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Initialize mAP calculator
map_metric = MeanAveragePrecision()

# Add predictions and targets to the mAP calculator
for pred, target in zip(nms_predictions, all_targets):
    pred_boxes = torch.stack([p['boxes'] for p in pred])
    pred_scores = torch.stack([p['scores'] for p in pred])
    pred_labels = torch.stack([p['labels'] for p in pred])

    target_boxes = torch.stack([t['boxes'] for t in target])
    target_labels = torch.stack([t['labels'] for t in target])

    map_metric.update(pred_boxes, pred_scores, pred_labels, target_boxes, target_labels)

# Compute mAP
map_value = map_metric.compute()
print(f"mAP: {map_value}")
